# PBIT Safety — CMR-targeted Hyperparameter Optimisation (LightGBM)

**Objective**: minimise CMR (Critical Miss Rate) — rate at which Sev3 faults (F03, F06)
are predicted as Sev0 (N01, N02).

**Constraint**: balanced accuracy ≥ 0.50 across CV folds (trials below return CMR = 1.0).

**Config**: Aug + Freq×Sev.

**Tunable parameters** (beyond standard LightGBM hyperparams):

| Parameter | Range | Purpose |
|-----------|-------|---------|
| `f05_boost` | 1–10× | F05 sample weight multiplier — maintains F05 recall |
| `sev3_boost` | 1–20× | F03 and F06 sample weight multiplier — forces model to avoid Sev3→Sev0 errors |

**Trials**: 100 (TPE sampler).  
Best params saved to `data/best_lgbm_cmr_params.json`.

## Setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks_safety':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from pbit.modeling import (
    META_COLS, FAULT_ORDER,
    make_cv_folds, summarize_cv,
    compute_sample_weights, evaluate_predictions, compute_safety_metrics,
    safety_results_table,
)
from pbit.data_validation import load_long_dataset
from pbit.augmentation import augment_training_set
from pbit.features import extract_features

TRAIN_PATH      = PROJECT_ROOT / 'data' / 'train_real.csv'
LONG_PATH       = PROJECT_ROOT / 'data' / 'PBIT_atuador_DESCARAC_r1_long_clean.csv'
F1_PARAMS_PATH  = PROJECT_ROOT / 'data' / 'best_lgbm_params.json'
CMR_PARAMS_PATH = PROJECT_ROOT / 'data' / 'best_lgbm_cmr_params.json'

PROJECT_ROOT

## Load Data & Folds

In [ ]:
train_df  = pd.read_csv(TRAIN_PATH)
long_df   = load_long_dataset(LONG_PATH)
feat_cols = [c for c in train_df.columns if c not in META_COLS]
folds     = make_cv_folds(train_df, n_splits=5, random_state=42)

print(f'Training set : {len(train_df):,} experiments x {len(feat_cols)} features')
print(f'Folds        : {len(folds)}')

## CMR Objective Function

Each trial trains LightGBM with Aug + Freq×Sev weighting across 5 CV folds.  
`sev3_boost` multiplies F03 and F06 sample weights — high values penalise Sev3→Sev0 errors  
during training, directly targeting CMR reduction.

In [ ]:
def cmr_objective(trial: optuna.Trial) -> float:
    params = {
        'n_estimators':      trial.suggest_int('n_estimators',      200, 800),
        'learning_rate':     trial.suggest_float('learning_rate',    0.01, 0.2,  log=True),
        'num_leaves':        trial.suggest_int('num_leaves',         15,  127),
        'min_child_samples': trial.suggest_int('min_child_samples',  3,   50),
        'subsample':         trial.suggest_float('subsample',        0.5,  1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5,  1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha',        1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda',       1e-8, 10.0, log=True),
        'random_state': 42,
        'verbose': -1,
    }
    f05_boost  = trial.suggest_float('f05_boost',  1.0, 10.0)
    sev3_boost = trial.suggest_float('sev3_boost', 1.0, 20.0)

    fold_cmrs, fold_bal, fold_f1s = [], [], []
    fold_f05s, fold_f06s, fold_down = [], [], []

    for fold_i, (tr_idx, val_idx) in enumerate(folds):
        tr_real  = train_df.iloc[tr_idx]
        val_real = train_df.iloc[val_idx]

        tr_exp_ids = set(tr_real['Experiment_Index'].tolist())
        aug_long   = augment_training_set(
            long_df, tr_exp_ids,
            target_per_class=160, noise_scale=0.01,
            random_state=42 + fold_i,
        )
        if len(aug_long) > 0:
            aug_feat = extract_features(aug_long)
            tr_fold  = pd.concat([tr_real, aug_feat], ignore_index=True)
        else:
            tr_fold = tr_real

        X_train = tr_fold[feat_cols].values.astype(float)
        y_train = tr_fold['Fault_Mode'].values
        X_val   = val_real[feat_cols].values.astype(float)
        y_val   = val_real['Fault_Mode'].values

        sw = compute_sample_weights(y_train, 'freq_sev')
        sw[y_train == 'F05'] *= f05_boost
        sw[np.isin(y_train, ['F03', 'F06'])] *= sev3_boost

        model = lgb.LGBMClassifier(**params)
        model.fit(X_train, y_train, sample_weight=sw)
        y_pred = model.predict(X_val)

        m  = evaluate_predictions(y_val, y_pred)
        sm = compute_safety_metrics(y_val, y_pred)
        fold_f1s.append(m['macro_f1'])
        fold_bal.append(m['balanced_accuracy'])
        fold_f05s.append(m['f05_recall'])
        fold_f06s.append(m['f06_recall'])
        fold_cmrs.append(sm['cmr'])
        fold_down.append(sm['down_risk'])

    mean_cmr  = float(np.nanmean([v for v in fold_cmrs if not np.isnan(v)] or [1.0]))
    mean_bal  = float(np.mean(fold_bal))
    mean_f1   = float(np.mean(fold_f1s))
    mean_f05  = float(np.nanmean([v for v in fold_f05s if not np.isnan(v)] or [0.0]))
    mean_f06  = float(np.nanmean([v for v in fold_f06s if not np.isnan(v)] or [0.0]))
    mean_down = float(np.mean(fold_down))

    trial.set_user_attr('macro_f1',   round(mean_f1,   4))
    trial.set_user_attr('bal_acc',    round(mean_bal,  4))
    trial.set_user_attr('f05_recall', round(mean_f05,  4))
    trial.set_user_attr('f06_recall', round(mean_f06,  4))
    trial.set_user_attr('down_risk',  round(mean_down, 4))

    if mean_bal < 0.50:
        return 1.0

    return mean_cmr

## Run CMR Study — 100 trials (~60–80 min)

In [ ]:
sampler   = optuna.samplers.TPESampler(seed=42)
cmr_study = optuna.create_study(direction='minimize', sampler=sampler)

t0 = time.time()
cmr_study.optimize(cmr_objective, n_trials=100, show_progress_bar=True)
elapsed = time.time() - t0

bt = cmr_study.best_trial
print(f'\nCompleted {len(cmr_study.trials)} trials in {elapsed/60:.1f} min')
print(f'Best CMR        : {cmr_study.best_value:.4f}')
print(f'  Macro F1      : {bt.user_attrs["macro_f1"]}')
print(f'  Balanced Acc  : {bt.user_attrs["bal_acc"]}')
print(f'  F06 recall    : {bt.user_attrs["f06_recall"]}')
print(f'  F05 recall    : {bt.user_attrs["f05_recall"]}')
print(f'  Downward Risk : {bt.user_attrs["down_risk"]}')
print(f'  sev3_boost    : {bt.params["sev3_boost"]:.3f}')
print(f'  f05_boost     : {bt.params["f05_boost"]:.3f}')
print('Best params:')
for k, v in bt.params.items():
    print(f'  {k}: {v}')

## Optimisation History

In [ ]:
trials_df = cmr_study.trials_dataframe()
feasible  = trials_df[trials_df['value'] < 1.0].copy()
infeas    = trials_df[trials_df['value'] >= 1.0]

print(f'Feasible trials (bal_acc >= 0.50): {len(feasible)} / {len(trials_df)}')
print(f'CMR range (feasible): {feasible["value"].min():.4f} – {feasible["value"].max():.4f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.scatter(feasible['number'], feasible['value'],
           alpha=0.6, s=20, color='steelblue', label='Feasible')
ax.scatter(infeas['number'], [0.5] * len(infeas),
           alpha=0.25, s=10, color='lightcoral', label='Infeasible')
ax.plot(feasible['number'], feasible['value'].cummin(),
        color='tomato', linewidth=1.5, label='Best so far')
ax.set_xlabel('Trial'); ax.set_ylabel('CMR')
ax.set_title('CMR optimisation history')
ax.legend(fontsize=8); ax.grid(True, linewidth=0.3)

ax = axes[1]
sc = ax.scatter(
    feasible['user_attrs_macro_f1'], feasible['value'],
    c=feasible['user_attrs_f06_recall'], cmap='RdYlGn', alpha=0.7, s=25,
)
plt.colorbar(sc, ax=ax, label='F06 recall')
ax.scatter(bt.user_attrs['macro_f1'], cmr_study.best_value,
           marker='*', s=200, color='gold', edgecolors='black', zorder=5, label='Best')
ax.set_xlabel('Macro F1'); ax.set_ylabel('CMR')
ax.set_title('CMR vs Macro F1 (colour = F06 recall)')
ax.legend(fontsize=8); ax.grid(True, linewidth=0.3)

ax = axes[2]
ax.scatter(feasible['params_sev3_boost'], feasible['value'],
           alpha=0.6, s=20, color='darkorange')
ax.scatter(bt.params['sev3_boost'], cmr_study.best_value,
           marker='*', s=200, color='gold', edgecolors='black', zorder=5, label='Best')
ax.set_xlabel('sev3_boost'); ax.set_ylabel('CMR')
ax.set_title('CMR vs sev3_boost')
ax.legend(fontsize=8); ax.grid(True, linewidth=0.3)

plt.tight_layout()
plt.show()

## Parameter Importance

In [ ]:
importance = optuna.importance.get_param_importances(cmr_study)

fig, ax = plt.subplots(figsize=(8, 4))
keys   = list(importance.keys())
vals   = list(importance.values())
colors = ['tomato' if k in ('sev3_boost', 'f05_boost') else 'steelblue' for k in keys]
ax.barh(keys[::-1], vals[::-1], color=colors[::-1])
ax.set_xlabel('Importance (fANOVA)')
ax.set_title('Hyperparameter importance for CMR objective')
ax.grid(True, axis='x', linewidth=0.3)
plt.tight_layout()
plt.show()

print('Parameter importances:')
for k, v in importance.items():
    print(f'  {k:<22}: {v:.4f}')

## Save CMR-tuned Parameters

In [ ]:
best_cmr = cmr_study.best_params.copy()
best_cmr['random_state'] = 42
best_cmr['verbose']      = -1

with open(CMR_PARAMS_PATH, 'w') as f:
    json.dump(best_cmr, f, indent=2)

print(f'Saved: {CMR_PARAMS_PATH}')
print(json.dumps(best_cmr, indent=2))

## CV Comparison: F1-tuned vs CMR-tuned

Re-run full 5-fold CV with CMR-tuned params and compare against the F1-tuned model
(loaded from `best_lgbm_params.json`). Both use Aug + Freq×Sev config.

In [ ]:
def run_full_cv(params, f05_boost, sev3_boost=1.0, label=''):
    """5-fold CV with Aug + Freq*Sev, optional sev3_boost."""
    fold_results = []
    fold_cmrs, fold_down, fold_severr = [], [], []

    for fold_i, (tr_idx, val_idx) in enumerate(folds):
        tr_real  = train_df.iloc[tr_idx]
        val_real = train_df.iloc[val_idx]
        tr_exp_ids = set(tr_real['Experiment_Index'].tolist())
        aug_long   = augment_training_set(
            long_df, tr_exp_ids,
            target_per_class=160, noise_scale=0.01, random_state=42 + fold_i,
        )
        if len(aug_long) > 0:
            aug_feat = extract_features(aug_long)
            tr_fold  = pd.concat([tr_real, aug_feat], ignore_index=True)
        else:
            tr_fold = tr_real

        X_train = tr_fold[feat_cols].values.astype(float)
        y_train = tr_fold['Fault_Mode'].values
        X_val   = val_real[feat_cols].values.astype(float)
        y_val   = val_real['Fault_Mode'].values

        sw = compute_sample_weights(y_train, 'freq_sev')
        sw[y_train == 'F05'] *= f05_boost
        if sev3_boost > 1.0:
            sw[np.isin(y_train, ['F03', 'F06'])] *= sev3_boost

        model = lgb.LGBMClassifier(**params)
        model.fit(X_train, y_train, sample_weight=sw)
        y_pred = model.predict(X_val)

        m  = evaluate_predictions(y_val, y_pred)
        sm = compute_safety_metrics(y_val, y_pred)
        m['fold'] = fold_i
        fold_results.append(m)
        fold_cmrs.append(sm['cmr'])
        fold_down.append(sm['down_risk'])
        fold_severr.append(sm['sev_err'])

    s = summarize_cv(fold_results)
    s['cmr_cv']     = float(np.nanmean([v for v in fold_cmrs   if not np.isnan(v)] or [float('nan')]))
    s['down_cv']    = float(np.mean(fold_down))
    s['severr_cv']  = float(np.mean(fold_severr))
    return s


# Load F1-tuned params
with open(F1_PARAMS_PATH) as f:
    f1_all = json.load(f)
f1_f05_boost = f1_all.pop('f05_boost')
f1_params    = f1_all

# Load CMR-tuned params
with open(CMR_PARAMS_PATH) as f:
    cmr_all = json.load(f)
cmr_sev3_boost = cmr_all.pop('sev3_boost')
cmr_f05_boost  = cmr_all.pop('f05_boost')
cmr_params     = cmr_all

print('Running F1-tuned CV...')
t0 = time.time()
s_f1  = run_full_cv(f1_params,  f1_f05_boost,  sev3_boost=1.0, label='F1-tuned')
print(f'Done in {time.time()-t0:.0f}s')

print('Running CMR-tuned CV...')
t0 = time.time()
s_cmr = run_full_cv(cmr_params, cmr_f05_boost, sev3_boost=cmr_sev3_boost, label='CMR-tuned')
print(f'Done in {time.time()-t0:.0f}s')

## Comparison Table

In [ ]:
print('=== CV comparison: F1-tuned vs CMR-tuned (Aug + Freq×Sev) ===')
print()
print(f"{'Metric':<25} {'F1-tuned':>12} {'CMR-tuned':>12} {'Delta':>10}")
print('-' * 62)

rows = [
    ('Macro F1',        'macro_f1_mean'),
    ('Balanced Acc',    'balanced_accuracy_mean'),
    ('F06 Recall',      'f06_recall_mean'),
    ('F05 Recall',      'f05_recall_mean'),
    ('F04 Recall',      'f04_recall_mean'),
    ('Sev3 Recall',     'sev3_recall_mean'),
]
for label, key in rows:
    v1 = s_f1.get(key, float('nan'))
    v2 = s_cmr.get(key, float('nan'))
    delta = v2 - v1
    print(f"{label:<25} {v1:>12.4f} {v2:>12.4f} {delta:>+10.4f}")

print()
print(f"{'Safety metric':<25} {'F1-tuned':>12} {'CMR-tuned':>12} {'Delta':>10}")
print('-' * 62)
for label, k1, k2 in [
    ('CMR (CV mean)',      'cmr_cv',    'cmr_cv'),
    ('Downward Risk',      'down_cv',   'down_cv'),
    ('Severity Error',     'severr_cv', 'severr_cv'),
]:
    v1 = s_f1.get(k1, float('nan'))
    v2 = s_cmr.get(k2, float('nan'))
    delta = v2 - v1
    print(f"{label:<25} {v1:>12.4f} {v2:>12.4f} {delta:>+10.4f}")

print()
print(f'sev3_boost used in CMR-tuned: {cmr_sev3_boost:.3f}')
print(f'f05_boost  used in CMR-tuned: {cmr_f05_boost:.3f}')